# M1 — TMDB data acquisition
Kseniia Tikhonova, Anastasiia German

**Scope:** movies released 2000–2024 with at least 50 votes: details, genres, keywords,
directors and top-5 billed cast (person IDs and names only).

**Output:** `movies.parquet` (one row per movie) and `acquisition_log.json`.

## 1. Setup

In [7]:
import hashlib, json, os, random, time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

In [8]:
SCOPE = ("TMDB: movies released 2000-2024 with at least 50 votes; details, genres, "
         "keywords, directors and top-5 billed cast (person IDs and names only).")
SCOPE_REVISION = None   # if we change the scope at M1: new scope + one sentence why

BASE = "https://api.themoviedb.org/3"
UA = "datacq-m1-tmdb (kseniia.tikhonova@etu.univ-grenoble-alpes.fr)"
TEST_PAGES = None       # 1 = quick test (1 page per year), None = full run
POLITE_DELAY = 0.25     # seconds between requests

DISCOVER = {"vote_count.gte": 50, "include_adult": "false", "include_video": "false",
            "sort_by": "primary_release_date.asc", "language": "en-US"}
DETAILS = {"append_to_response": "keywords,credits", "language": "en-US"}

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)

The key is read from the `.env` file

In [9]:
load_dotenv()   # reads TMDB_API_KEY from .env
KEY = os.environ["TMDB_API_KEY"].strip()

# read access token (starts with "eyJ") goes in a header, short v3 key goes in the URL
if KEY.startswith("eyJ"):
    HEADERS = {"User-Agent": UA, "Authorization": f"Bearer {KEY}"}
    AUTH = {}
else:
    HEADERS = {"User-Agent": UA}
    AUTH = {"api_key": KEY}

## 2. HTTP helpers
`http_get` retries on 429/5xx with exponential backoff + jitter.
`cached_get` saves every response to `cache/`, so a crashed run can be resumed.

In [10]:
def http_get(url, params, attempts=5):
    for k in range(attempts):
        wait = random.uniform(0, 2 ** k)
        try:
            r = requests.get(url, params={**params, **AUTH}, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                return r.json()
            if r.status_code not in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"{r.status_code} for {url}")   # no point retrying
            wait = int(r.headers.get("Retry-After", 0)) or wait
        except (requests.ConnectionError, requests.Timeout):
            pass
        time.sleep(wait)
    raise RuntimeError(f"gave up on {url}")

In [11]:
requests_made = 0

def cached_get(url, params):
    global requests_made
    key = hashlib.sha256((url + json.dumps(params, sort_keys=True)).encode()).hexdigest()[:16]
    path = CACHE / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    data = http_get(url, params)
    tmp = path.with_suffix(".tmp")   # write to a temp file first, so an interrupted
    tmp.write_text(json.dumps(data), encoding="utf-8")
    tmp.replace(path)                # run never leaves a broken file in the cache
    requests_made += 1
    time.sleep(POLITE_DELAY)
    return data

## 3. Step 1 — movie IDs (`/discover/movie`)
One query per year: TMDB returns at most 500 pages (10,000 results) per query.

In [12]:
time_started = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())

ids, per_year = [], {}
for year in range(2000, 2025):
    params = {**DISCOVER, "primary_release_date.gte": f"{year}-01-01",
              "primary_release_date.lte": f"{year}-12-31"}
    page, got = 1, 0
    while True:
        data = cached_get(f"{BASE}/discover/movie", {**params, "page": page})
        ids += [m["id"] for m in data["results"]]
        got += len(data["results"])
        if not data["results"] or page >= min(data["total_pages"], 500, TEST_PAGES or 500):
            break
        page += 1
    per_year[year] = {"total_results": data["total_results"], "collected": got}
    print(year, got, "/", data["total_results"])

ids = list(dict.fromkeys(ids))   # remove duplicates
print("unique movies:", len(ids))

2000 414 / 414
2001 477 / 477
2002 493 / 493
2003 535 / 535
2004 590 / 590
2005 614 / 615
2006 775 / 775
2007 772 / 772
2008 837 / 837
2009 911 / 911
2010 907 / 907
2011 965 / 965
2012 987 / 987
2013 1129 / 1129
2014 1221 / 1221
2015 1274 / 1274
2016 1339 / 1339
2017 1415 / 1415
2018 1441 / 1441
2019 1419 / 1419
2020 1101 / 1101
2021 1137 / 1137
2022 1178 / 1178
2023 1094 / 1094
2024 939 / 939
unique movies: 23964


## 4. Step 2 — details, keywords and credits (`/movie/{id}`)
`append_to_response` gives all three in one request. One row per movie; we keep only
directors and the top-5 cast (IDs and names) to minimize personal data.

In [13]:
rows, failed = [], []
for i, mid in enumerate(ids, 1):
    try:
        d = cached_get(f"{BASE}/movie/{mid}", DETAILS)
    except (requests.HTTPError, RuntimeError) as e:
        failed.append({"tmdb_id": mid, "error": str(e)})
        continue
    directors = {p["id"]: p["name"] for p in d["credits"]["crew"] if p["job"] == "Director"}
    cast = sorted(d["credits"]["cast"], key=lambda p: p["order"])[:5]
    rows.append({
        "tmdb_id": d["id"], "title": d["title"], "release_date": d["release_date"],
        "original_language": d["original_language"], "runtime": d["runtime"],
        "budget": d["budget"], "revenue": d["revenue"],
        "vote_average": d["vote_average"], "vote_count": d["vote_count"],
        "popularity": d["popularity"],
        "genres": [g["name"] for g in d["genres"]],
        "keywords": [k["name"] for k in d["keywords"]["keywords"]],
        "director_ids": list(directors.keys()),
        "director_names": list(directors.values()),
        "cast_ids": [p["id"] for p in cast],
        "cast_names": [p["name"] for p in cast],
    })
    if i % 500 == 0:
        print(i, "/", len(ids))

print("failed:", len(failed))

KeyboardInterrupt: 

In [ ]:
movies = pd.DataFrame(rows)
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
assert movies["tmdb_id"].is_unique
movies.head()

## 5. Save as Parquet

In [ ]:
movies.to_parquet("movies.parquet", index=False, compression="zstd")

back = pd.read_parquet("movies.parquet")
print("movies.parquet shape:", back.shape)
back.dtypes

## 6. Acquisition log

In [ ]:
log = {
    "url": {"discover": f"{BASE}/discover/movie", "details": f"{BASE}/movie/{{id}}"},
    "parameters": {
        "discover": {**DISCOVER, "years": "2000-2024, one query per year",
                     "pages": TEST_PAGES or "all (max 500)"},
        "details": DETAILS,
    },
    "auth": "api_key in query string" if AUTH else "bearer token in header",
    "num_found": sum(v["total_results"] for v in per_year.values()),
    "time_started": time_started,
    "time_ended": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "api_version": 3,
    "scope": SCOPE,
    "scope_revision": SCOPE_REVISION,
    "encoding": "utf-8",
    "per_year": per_year,
    "unique_movies": len(ids),
    "failed": failed,
    "http_requests": requests_made,
    "parquet_shape": {"movies.parquet": list(back.shape)},
    "path_to_file": "movies.parquet",
    "path_to_cache": "cache/",
}
Path("acquisition_log.json").write_text(json.dumps(log, indent=2, ensure_ascii=False),
                                        encoding="utf-8")
print(json.dumps(log, indent=2, ensure_ascii=False))